# 情報数学Ⅲ 第15回

### 🔍 利用方法

- このノートブックは**閲覧専用**です。  
  自分のGoogleドライブにコピーを作成してから編集してください。

  1. メニューの「**ファイル → ドライブにコピーを保存**」を選ぶ  
  2. コピーしたノートブックの**ファイル名（左上の"xxxx"）を学籍番号に変更**してください（提出時の識別のため）

---

### 🔧 提出手順

1. Google Colab の右上にある「**共有**」ボタンをクリック  
2. 「**一般的なアクセス**」の設定を「**リンクを知っている全員**」に変更  
3. アクセス権を「**閲覧者**」に設定（⚠️「編集者」にしないこと！）  
4. 表示されるURLをコピー  
5. WebClassの提出フォームに、その**URLを貼り付けて提出**

---

### 💡 補足・注意点

- **「編集者」ではなく「閲覧者」**に設定してください。  
  → 教員が間違ってファイルを上書きしてしまうのを防ぐためです。
- **提出したリンクを自分でも一度開いてみて、ちゃんと共有されているかを確認**してください。

In [ ]:
# 使用するライブラリのインポート
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import numpy as np

import pymc as pm
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import mean_squared_error

# 日本語表示: Colab環境用
#!pip install japanize_matplotlib
#import japanize_matplotlib

# 日本語表示: 教員確認用（Mac）
# 日本語フォントを指定（ヒラギノ角ゴがmacOSに標準搭載）
plt.rcParams['font.family'] = 'Hiragino Sans'


## ロジスティック回帰

* 教科書付属のノートブック（9-2-ロジスティック回帰.ipynb）を参照すること

### ロジスティック関数とロジット関数


In [ ]:
# ロジスティック関数（シグモイド）: sigmoid(x) = 1 / (1 + exp(-x))
def logistic(x):
    return 1 / (1 + np.exp(-x))

# ロジット関数: logit(p) = log(p / (1 - p))
def logit(p):
    return np.log(p / (1 - p))

# プロット用の範囲
x_vals = np.linspace(-10, 10, 400)
p_vals = np.linspace(0.001, 0.999, 400)  # 0と1は避ける（logitは発散する）

# 描画
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 左: ロジスティック関数
axes[0].plot(x_vals, logistic(x_vals), color='blue')
axes[0].set_title('ロジスティック関数（シグモイド）')
axes[0].set_xlabel('x')
axes[0].set_ylabel('σ(x) = 1 / (1 + exp(-x))')
axes[0].grid(True)

# 右: ロジット関数
axes[1].plot(p_vals, logit(p_vals), color='red')
axes[1].set_title('ロジット関数（logit）')
axes[1].set_xlabel('p')
axes[1].set_ylabel('log(p / (1 - p))')
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 参考：PyMCを使ったベイズ推定
* 線形回帰でベイズ推定を使う
* 前にやった最小二乗法による線形回帰と比較してみる

In [ ]:
# mtcarsデータセットの読み込み
data = sm.datasets.get_rdataset("mtcars").data
data

In [ ]:
# wtとmpgの散布図を描画
plt.figure(figsize=(8, 6))
plt.scatter(data['wt'], data['mpg'], color='blue', alpha=0.5)
plt.title('wtとmpgの散布図')
plt.xlabel('wt (重量)')
plt.ylabel('mpg (燃費)')
plt.grid()
plt.show() 

## まずは最小二乗法でやってみよう

In [ ]:
# statsmodelsを使用して線形回帰モデルを作成
model = smf.ols('mpg ~ wt', data=data).fit()  
# モデルの要約を表示
model_summary = model.summary()
print(model_summary)

In [ ]:
# 上記の結果を可視化する
plt.figure(figsize=(8, 6))
plt.scatter(data['wt'], data['mpg'], color='blue', alpha=0.5, label='データポイント')
plt.plot(data['wt'], model.fittedvalues, color='red', label='回帰直線')
plt.title('wtとmpgの線形回帰')
plt.xlabel('wt (重量)')
plt.ylabel('mpg (燃費)')
plt.legend()
plt.grid()
plt.show()

## ベイズ推定を用いて線形回帰を行う
* PyMCを用いる
* 事前分布
  * 回帰定数・切片：正規分布
  * 誤差（イプシロン）：

In [ ]:
# PyMCのモデルを定義
with pm.Model() as model_bayes:
    # 事前分布の設定
    alpha = pm.Normal('alpha', mu=0, sigma=10) # 切片
    beta = pm.Normal('beta', mu=0, sigma=10) # 傾き
    epsilon = pm.HalfNormal('sigma', sigma=1) # 誤差の標準偏差

    # 線形モデルの定義
    #mu = alpha + beta * data['wt'] # トレースを残さないとき（推定値を可視化しない場合）
    mu = pm.Deterministic("mu", alpha + beta * data['wt']) # トレースを残すとき（推定値を可視化するため）

    # 尤度関数の定義（観測されたデータ）
    Y_obs = pm.Normal('Y_obs', mu=mu, sigma=epsilon, observed=data['mpg'])


In [ ]:
# モデルを確認
model_bayes

In [ ]:
# 観測されている確率変数
model_bayes.observed_RVs

In [ ]:
# 観測されていない確率変数
model_bayes.unobserved_RVs

In [ ]:
# サンプリングの実行
with model_bayes:
    trace = pm.sample(random_seed=0) 

### トレースプロットの表示
* 得られたパラメータの分布と，サンプリングの状況
* 状況が収束しているかの確認にも用いる

In [ ]:
# トレースプロットを表示
names = ['alpha', 'beta', 'sigma'] # muは表示させない
pm.plot_trace(trace, var_names=names)

In [ ]:
# 要約統計量を表示
pm.summary(trace, var_names=names)

### 回帰直線とその不確実性を可視化

- 青：観測データ（車の重量 vs 燃費）
- 赤い直線：**平均的な回帰直線**
- 赤い帯：**±1σの不確実性**（予測分布の広がり）

### ベイズ回帰の特徴

- 回帰係数（切片・傾き・誤差）が確率分布として求まる
- 1本の「確定した直線」ではなく、**不確実性を含めた予測**ができる
- 平均直線だけでなく、その上下のばらつき（信頼帯）も重要


In [ ]:
# 回帰直線とその不確実性を可視化
plt.figure(figsize=(8, 6))
plt.scatter(data['wt'], data['mpg'], color='blue', alpha=0.5, label='データポイント')
x_vals = np.linspace(data['wt'].min(), data['wt'].max(), 100)

alpha_mean = trace.posterior['alpha'].mean().values # 参考書のようにtrace['alpha']とするとエラー（PyMC3ではOK）
beta_mean = trace.posterior['beta'].mean().values
sigma_mean = trace.posterior['sigma'].mean().values

y_vals = alpha_mean + beta_mean * x_vals

plt.plot(x_vals, y_vals, color='red', label='回帰直線 (平均)')
plt.fill_between(x_vals,
                 y_vals - sigma_mean,
                 y_vals + sigma_mean,
                 color='red', alpha=0.2, label='不確実性 (±1σ)')
plt.title('wtとmpgの線形回帰（ベイズ推定）')
plt.xlabel('wt (重量)')
plt.ylabel('mpg (燃費)')
plt.legend()
plt.grid()
plt.show() 

## 演習

上記のmtcarsデータセットを用いて，何か変数を選んでロジスティック回帰を実行してみよ
* 例）燃費の高い車はMTになりやすいか？

### 参考：mtcars データセットの変数一覧

| 変数名 | 説明                     | 単位・内容                     |
|--------|--------------------------|-------------------------------|
| mpg    | 燃費                     | miles per gallon（mpg）       |
| cyl    | シリンダー数             | 4, 6, 8                       |
| disp   | 排気量                   | 立方インチ                    |
| hp     | 馬力                     | horsepower                    |
| drat   | リアアクスル比           | 比率                          |
| wt     | 車体重量                 | 1000ポンド単位               |
| qsec   | 1/4マイル加速時間        | 秒                            |
| vs     | エンジン（0=V型, 1=直列） | 二値                         |
| am     | 変速機（0=AT, 1=MT）      | 二値                         |
| gear   | 前進ギア数               | 3, 4, 5                       |
| carb   | キャブレター数           | 台数                          |

---